<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color='#BFD72F'>**Methodology** </font> <a class="anchor" id='top'></a> 

- [1. Introduction](#1) 
- [2. Import Libraries](#2) 
- [3. Create Metadata](#3)
- [4. Import Dataset](#4)  
- [5. Model and Assessment](#5) 
    - [5.1 Linear Regression](#5_1)
        - [5.1.1 Ordinary Least Squares (OLS)](#5_1_1)
        - [5.1.2 Ridge Regression](#5_1_2)
        - [5.1.3 Lasso Regression](#5_1_3)
        - [5.1.4 Elastic Net Regression](#5_1_4)
        - [5.1.5 Random Forest](#5_1_5)
    - [5.2 Model Comparison](#5_2)
    - [5.3 Test Models](#5_3)
- [6. Save the Results to Kaggle](#6) 
- [7. End of the Notebook](#7) 

<a class="anchor" id="1">

# **1. Introduction**

[Back to TOP](#TOP)
</a>

Here we will create models to predict the used car prices. We will create several models and later analyse their performances and choose the one that best predicts the target feature based on the MAE metric. The results here represent our final models with the thresholds and features defined in the "03_Feature_Selection" file that yield the best results we could.

<a class="anchor" id="2">

# **2. Import libraries**

[Back to TOP](#TOP)
</a>

The following libraries will help us develop the analyses and model for this project. Pandas and Numpy, provide the efficient tools for data manipulation, cleaning and numerical computations. Matplotlib and Seaborn are used to create clear and informative visualizations. Finally, Scikit-learn offers a range of Machine Learning tools for model training, spliting the data and evaluate model performance. 

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import os


from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

#KNN
from sklearn.neighbors import KNeighborsRegressor


#Model evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, mean_absolute_percentage_error
import statsmodels.api as sm

# Load Created Functions
from visualizations import *
from data_preprocessing import *
from model_and_assessment import *

# Set random seed for reproducibility
np.random.seed(40111) 

<a class="anchor" id="3">

# **3. Create Metadata**

[Back to TOP](#TOP)
</a>

Understanding the features helps interpret the data correctly and supports subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



<a class="anchor" id="4">

# **4. Import Dataset**

[Back to TOP](#top)
</a>

In this section, we load the preprocessed datasets that were previously saved after the data preprocessing steps. These datasets are ready for modelling and to get results.

In [2]:
# Define relative path to the preprocessed data folder (outside "notebooks/")
data_path = "../data_feature_selected/"

# Load preprocessed datasets
X_train = pd.read_csv(f"{data_path}X_train_final.csv", index_col=0)
X_val   = pd.read_csv(f"{data_path}X_val_final.csv", index_col=0)
test    = pd.read_csv(f"{data_path}test_final.csv", index_col=0)

# Load target variables and squeeze to convert DataFrame to Series since they have only one column
y_train = pd.read_csv(f"{data_path}y_train.csv", index_col=0).squeeze()
y_val   = pd.read_csv(f"{data_path}y_val.csv", index_col=0).squeeze()

<a class="anchor" id="5">

# **5. Model and Assessment**

[Back to TOP](#TOP)
</a>

In this notebook, we focus on training and evaluating different regression models to predict car resale prices. We start with a simple model to establish a baseline and progressively explore more complex models. The purpose of this notebook section is to create a structured workflow that allows us to train models efficiently, evaluate their performance, and compare predictions with actual values.

In [3]:
# Combine train and validation datasets
X_combined = np.concatenate([X_train, X_val])
y_combined = np.concatenate([y_train, y_val])

# Create a test fold index (-1 for train, 0 for validation)
test_fold = [-1] * len(X_train) + [0] * len(X_val)

print('Test fold: ', len(test_fold))
print('X_combined: ', len(X_combined))
print('y_combined: ', len(y_combined))

# Define the PredefinedSplit
ps = PredefinedSplit(test_fold=test_fold) # aqui diz que os dados de treino são os que têm label -1 e os de validação são os que têm label 0

Test fold:  75973
X_combined:  75973
y_combined:  75973


### Create a Sample to Test Models

In [4]:
X_train_s = X_train.sample(1000, random_state=42)
y_train_s = y_train.loc[X_train_s.index]
X_val_s = X_val.sample(500, random_state=42)
y_val_s = y_val.loc[X_val_s.index]

# Recriar combined sample
sample_X_combined = np.concatenate([X_train_s, X_val_s])
sample_y_combined = np.concatenate([y_train_s, y_val_s])

# Novo fold correto!
sample_test_fold = [-1] * len(X_train_s) + [0] * len(X_val_s)

ps_sample = PredefinedSplit(sample_test_fold)

## Apply Predefined Split with KNN

In [11]:
model_KNN = KNeighborsRegressor()
param_grid = {
    "n_neighbors": [10, 20, 30, 40, 50, 80, 100],              # 3..80
    "weights": ["uniform", "distance"],
    "p": [1, 2],                                # Manhattan vs Euclidiana
    "leaf_size": [2, 5, 8, 12, 17, 25, 30, 40, 50, 60, 70, 80, 90, 100],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
}

scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

## **Primeiro Teste do KNN com uma samples para perceber se o código está correto e faz sentido**

In [13]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid, n_iter=10, scoring=scoring, refit='mae', verbose=1, cv=ps_sample, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(sample_X_combined, sample_y_combined)

# Get results dictionary
results = rsCV_KNN.cv_results_

# --- Extract scores for R2 ---
mean_train_r2 = results['mean_train_r2']
mean_val_r2   = results['mean_test_r2']

# --- Extract scores for MAE ---
# Atenção: ainda vêm como NEG-MAE, por isso aplicamos -
mean_train_mae = -results['mean_train_mae']
mean_val_mae   = -results['mean_test_mae']

# Extract parameters
parameters = results['params']

# Print each candidate with R2 + MAE + gap
for r2_t, r2_v, mae_t, mae_v, params in zip(mean_train_r2, mean_val_r2,
                                            mean_train_mae, mean_val_mae, parameters):
    print(
        f"Train R2={r2_t:.3f} | Val R2={r2_v:.3f} | "
        f"Train MAE={mae_t:.1f} | Val MAE={mae_v:.1f} | "
        f"Gap R2={r2_t - r2_v:.3f} | Params={params}"
    )

# --- Best models based on refit metric (MAE) ---
best_idx = rsCV_KNN.best_index_

best_train_r2 = results['mean_train_r2'][best_idx]
best_val_r2   = results['mean_test_r2'][best_idx]

best_train_mae = -results['mean_train_mae'][best_idx]
best_val_mae   = -results['mean_test_mae'][best_idx]

best_params = results['params'][best_idx]

print("\n=== BEST MODEL (based on MAE) ===")
print("Best train R2:", best_train_r2)
print("Best validation R2:", best_val_r2)
print("Best train MAE:", best_train_mae)
print("Best validation MAE:", best_val_mae)
print("R2 Gap:", (best_train_r2 - best_val_r2) / best_train_r2 * 100, "%")
print("Best parameters:", best_params)


# O QUE ESTÁ EM COMENTÁRIO ABAIXO VEIO DA AULA DO STOR

# # All results
# means = rsCV_KNN.cv_results_['mean_test_score']
# stds = rsCV_KNN.cv_results_['std_test_score']    
# for mean, std, params in zip(means, stds, rsCV_KNN.cv_results_['params']):
#     print("%0.3f (+/-%0.03f) for %r" % (mean, std , params))

Fitting 1 folds for each of 10 candidates, totalling 10 fits
Train R2=1.000 | Val R2=0.858 | Train MAE=-0.0 | Val MAE=2519.8 | Gap R2=0.142 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 10, 'leaf_size': 60, 'algorithm': 'auto'}
Train R2=1.000 | Val R2=0.832 | Train MAE=-0.0 | Val MAE=2655.0 | Gap R2=0.168 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 20, 'leaf_size': 70, 'algorithm': 'auto'}
Train R2=0.802 | Val R2=0.768 | Train MAE=2966.0 | Val MAE=3151.5 | Gap R2=0.034 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 30, 'leaf_size': 90, 'algorithm': 'kd_tree'}
Train R2=1.000 | Val R2=0.793 | Train MAE=-0.0 | Val MAE=2876.5 | Gap R2=0.207 | Params={'weights': 'distance', 'p': 1, 'n_neighbors': 40, 'leaf_size': 60, 'algorithm': 'brute'}
Train R2=1.000 | Val R2=0.710 | Train MAE=-0.0 | Val MAE=3372.6 | Gap R2=0.290 | Params={'weights': 'distance', 'p': 2, 'n_neighbors': 100, 'leaf_size': 50, 'algorithm': 'kd_tree'}
Train R2=0.785 | Val R2=0.743 | Train MAE=306

## **Testar Novamente KNN com o dataset todo**

In [ ]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid, n_iter=10, scoring=scoring, refit='mae', verbose=1, cv=ps, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(X_combined, y_combined)

# Get results dictionary
results = rsCV_KNN.cv_results_

# --- Extract scores for R2 ---
mean_train_r2 = results['mean_train_r2']
mean_val_r2   = results['mean_test_r2']

# --- Extract scores for MAE ---
# Atenção: ainda vêm como NEG-MAE, por isso aplicamos -
mean_train_mae = -results['mean_train_mae']
mean_val_mae   = -results['mean_test_mae']

# Extract parameters
parameters = results['params']

# Print each candidate with R2 + MAE + gap
for r2_t, r2_v, mae_t, mae_v, params in zip(mean_train_r2, mean_val_r2,
                                            mean_train_mae, mean_val_mae, parameters):
    print(
        f"Train R2={r2_t:.3f} | Val R2={r2_v:.3f} | "
        f"Train MAE={mae_t:.1f} | Val MAE={mae_v:.1f} | "
        f"Gap R2={r2_t - r2_v:.3f} | Params={params}"
    )

# --- Best models based on refit metric (MAE) ---
best_idx = rsCV_KNN.best_index_

best_train_r2 = results['mean_train_r2'][best_idx]
best_val_r2   = results['mean_test_r2'][best_idx]

best_train_mae = -results['mean_train_mae'][best_idx]
best_val_mae   = -results['mean_test_mae'][best_idx]

best_params = results['params'][best_idx]

print("\n=== BEST MODEL (based on MAE) ===")
print("Best train R2:", best_train_r2)
print("Best validation R2:", best_val_r2)
print("Best train MAE:", best_train_mae)
print("Best validation MAE:", best_val_mae)
print("R2 Gap:", (best_train_r2 - best_val_r2) / best_train_r2 * 100, "%")
print("Best parameters:", best_params)


# O QUE ESTÁ EM COMENTÁRIO ABAIXO VEIO DA AULA DO STOR

# # All results
# means = rsCV_KNN.cv_results_['mean_test_score']
# stds = rsCV_KNN.cv_results_['std_test_score']    
# for mean, std, params in zip(means, stds, rsCV_KNN.cv_results_['params']):
#     print("%0.3f (+/-%0.03f) for %r" % (mean, std , params))

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train=0.876 | Val=0.872 | Gap=0.004 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 40, 'leaf_size': 25, 'algorithm': 'auto'}
Train=0.999 | Val=0.878 | Gap=0.121 | Params={'weights': 'distance', 'p': 2, 'n_neighbors': 100, 'leaf_size': 90, 'algorithm': 'kd_tree'}
Train=0.912 | Val=0.891 | Gap=0.021 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 10, 'leaf_size': 2, 'algorithm': 'ball_tree'}
Train=0.999 | Val=0.888 | Gap=0.111 | Params={'weights': 'distance', 'p': 2, 'n_neighbors': 50, 'leaf_size': 2, 'algorithm': 'brute'}
Train=0.884 | Val=0.878 | Gap=0.006 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 30, 'leaf_size': 2, 'algorithm': 'auto'}
Train=0.894 | Val=0.889 | Gap=0.005 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 30, 'leaf_size': 8, 'algorithm': 'ball_tree'}
Train=0.871 | Val=0.869 | Gap=0.003 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 50, 'leaf_size': 40, 'algorithm': 'br

KeyError: 'mean_test_MAE'

Ao que parece se usarmos 'distance' overfitta sempre. Agora vou testar o mesmo modelo mas tirando 'distance' como opção

In [4]:
model_KNN = KNeighborsRegressor()
param_grid = {
    "n_neighbors": [10, 20, 30, 40, 50, 80, 100],              # 3..80
    "weights": ["uniform"],
    "p": [1, 2],                                # Manhattan vs Euclidiana
    "leaf_size": [2, 5, 8, 12, 17, 25, 30, 40, 50, 60, 70, 80, 90, 100],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
}

scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

In [6]:
rsCV_KNN = RandomizedSearchCV(model_KNN, param_grid, n_iter=15, scoring=scoring, refit='mae', verbose=1, cv=ps, return_train_score=True) 
# with verbose=1, we can see the progress of the search
# return_train_score=True to get training scores as well and understand overfitting

rsCV_KNN.fit(X_combined, y_combined)

# Get results dictionary
results = rsCV_KNN.cv_results_

# --- Extract scores for R2 ---
mean_train_r2 = results['mean_train_r2']
mean_val_r2   = results['mean_test_r2']

# --- Extract scores for MAE ---
# Atenção: ainda vêm como NEG-MAE, por isso aplicamos -
mean_train_mae = -results['mean_train_mae']
mean_val_mae   = -results['mean_test_mae']

# Extract parameters
parameters = results['params']

# Print each candidate with R2 + MAE + gap
for r2_t, r2_v, mae_t, mae_v, params in zip(mean_train_r2, mean_val_r2,
                                            mean_train_mae, mean_val_mae, parameters):
    print(
        f"Train R2={r2_t:.3f} | Val R2={r2_v:.3f} | "
        f"Train MAE={mae_t:.1f} | Val MAE={mae_v:.1f} | "
        f"Gap R2={r2_t - r2_v:.3f} | Params={params}"
    )

# --- Best models based on refit metric (MAE) ---
best_idx = rsCV_KNN.best_index_

best_train_r2 = results['mean_train_r2'][best_idx]
best_val_r2   = results['mean_test_r2'][best_idx]

best_train_mae = -results['mean_train_mae'][best_idx]
best_val_mae   = -results['mean_test_mae'][best_idx]

best_params = results['params'][best_idx]

print("\n=== BEST MODEL (based on MAE) ===")
print("Best train R2:", best_train_r2)
print("Best validation R2:", best_val_r2)
print("Best train MAE:", best_train_mae)
print("Best validation MAE:", best_val_mae)
print("R2 Gap:", (best_train_r2 - best_val_r2) / best_train_r2 * 100, "%")
print("Best parameters:", best_params)


# O QUE ESTÁ EM COMENTÁRIO ABAIXO VEIO DA AULA DO STOR

# # All results
# means = rsCV_KNN.cv_results_['mean_test_score']
# stds = rsCV_KNN.cv_results_['std_test_score']    
# for mean, std, params in zip(means, stds, rsCV_KNN.cv_results_['params']):
#     print("%0.3f (+/-%0.03f) for %r" % (mean, std , params))

Fitting 1 folds for each of 15 candidates, totalling 15 fits
Train R2=0.894 | Val R2=0.883 | Train MAE=1797.5 | Val MAE=1920.9 | Gap R2=0.011 | Params={'weights': 'uniform', 'p': 2, 'n_neighbors': 20, 'leaf_size': 30, 'algorithm': 'ball_tree'}
Train R2=0.883 | Val R2=0.879 | Train MAE=1918.7 | Val MAE=1979.0 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 50, 'leaf_size': 5, 'algorithm': 'brute'}
Train R2=0.919 | Val R2=0.900 | Train MAE=1550.9 | Val MAE=1752.4 | Gap R2=0.019 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 10, 'leaf_size': 5, 'algorithm': 'brute'}
Train R2=0.867 | Val R2=0.864 | Train MAE=2082.2 | Val MAE=2125.5 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 100, 'leaf_size': 50, 'algorithm': 'ball_tree'}
Train R2=0.883 | Val R2=0.879 | Train MAE=1918.7 | Val MAE=1979.0 | Gap R2=0.003 | Params={'weights': 'uniform', 'p': 1, 'n_neighbors': 50, 'leaf_size': 5, 'algorithm': 'ball_tree'}
Train R2=0.884 | Val R2=0.878 | Tr

<a class="anchor" id="5_2">

## **5.2** Model Comparison

[Back to TOP](#TOP)
</a>

After training and evaluating each regression model individually, we now compare their performance using key metrics. This comparison allows us to understand which model generalizes best to unseen data and whether regularization techniques offer improvements over standard Linear Regression.
By analyzing these metrics side by side, we can identify the most reliable model for predicting car resale prices and make informed decisions about potential adjustments or further tuning.

In [14]:
# List of models and their trained instances
models = {
    'OLS': lr_model,
    'Ridge': ridge_model,
    'Lasso': lasso_model,
    'Elastic Net': elastic_model,
    'Random Forest': rf_model
}

#Evaluate each model on the validation set and store results
results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X_val, y_val)

# Convert the results dictionary into a DataFrame
# Metrics as rows, models as columns
metrics_df = pd.DataFrame(results)

# Display the metrics
metrics_df.round(2)

NameError: name 'lr_model' is not defined

We compared five regression models, OLS, Ridge, Lasso, Elastic Net, and Random Forest, on both training and validation datasets. From this comparison, we observe that OLS, Ridge, and Lasso all perform very similarly, explaining approximately **81% of the variance (R² ≈ 0.81)** with nearly identical MAE and RMSE values. These consistent errors indicate that all three models generalize well to unseen data, showing **no signs of overfitting**.

Regularization (Ridge and Lasso) does not provide noticeable improvement over OLS, which suggests that multicollinearity and irrelevant features are not major issues at this stage of the project.

Elastic Net regression shows slightly lower performance, which is likely due to its use of both L1 and L2 regularization without optimized hyperparameters. To improve Elastic Net results, **tuning alpha and l1_ratio would be necessary**.

In contrast, the **Random Forest model outperforms all linear models, achieving a much higher R² (0.94)** and significantly lower prediction errors (MAE, RMSE, MedAE). This indicates that the nonlinear relationships in the dataset are better captured by an ensemble tree-based model.

Overall, OLS remains a strong interpretable baseline, while **Random Forest currently delivers the best predictive performance**.

<a class="anchor" id="5_3">

## **5.3** Test Models

[Back to TOP](#TOP)
</a>

In [ ]:
# Predict using all models on the test set
test_predictions = {
    'OLS': lr_model.predict(test),
    'Ridge': ridge_model.predict(test),
    'Lasso': lasso_model.predict(test),
    'Elastic Net': elastic_model.predict(test),
    'Random Forest': rf_model.predict(test)
}

# Convert to a DataFrame for easy comparison
test_predictions_df = pd.DataFrame(test_predictions, index=test.index)

# Show the first few predictions
test_predictions_df.head(10)

,OLS,Ridge,Lasso,Elastic Net,Random Forest
carID,,,,,
89856,15675.309931,15675.431883,15681.276327,15741.812970,15633.820
106581,23491.272504,23491.280788,23488.566020,23479.828674,25231.115
80886,11758.446644,11758.423772,11760.503114,11769.625994,14069.645
100174,16217.376405,16217.418122,16215.760666,16201.245897,16756.615
81376,27754.103248,27754.171223,27755.529933,27793.555946,26671.225
85391,9321.146831,9321.155783,9321.098271,9317.663725,10134.835
82175,14544.465553,14544.421450,14544.864581,14571.427507,14450.230
95250,14882.576266,14882.615057,14887.608349,14927.481371,15360.355
85071,2913.809701,2913.689818,2913.582678,2912.563452,5206.325


In [ ]:
test_predictions

{'OLS': array([15675.30993131, 23491.27250402, 11758.44664405, ...,
        34957.21829907, 20329.126581  , 11727.91362101]),
 'Ridge': array([15675.43188322, 23491.28078779, 11758.42377235, ...,
        34957.23190254, 20329.1853222 , 11727.92223941]),
 'Lasso': array([15681.27632663, 23488.5660196 , 11760.50311385, ...,
        34953.73288649, 20331.51321056, 11728.70557603]),
 'Elastic Net': array([15741.81297017, 23479.82867433, 11769.62599362, ...,
        34912.1102595 , 20340.53958088, 11738.45922989]),
 'Random Forest': array([15633.82 , 25231.115, 14069.645, ..., 34566.01 , 21085.345,
        10998.05 ])}

<a class="anchor" id="6">

# **6. Save results to Kaggle**

[Back to TOP](#TOP)
</a>

In [ ]:
# Go one level up from the notebooks folder to reach the repo root
selected_dir = "../results/kaggle_submissions/"
os.makedirs(selected_dir, exist_ok=True)

In [ ]:
best_model_name = metrics_df.loc['MAE'].idxmin()  # gets the column (model) with lowest MAE
best_mae_value = metrics_df.loc['MAE', best_model_name]

print(f"Best model based on MAE: {best_model_name} (MAE = {best_mae_value:.2f})")

Best model based on MAE: Random Forest (MAE = 1428.15)


In [ ]:
# Extract carID
car_ids = test.index.values

# Create DataFrame with best model predictions
best_model_df = pd.DataFrame({
    'carID': car_ids,
    'price': test_predictions[best_model_name]
})

# Save predictions to CSV
best_model_df.to_csv(f"{selected_dir}/{best_model_name.lower().replace(' ', '_')}_predictions.csv", index=False)


<a class="anchor" id="7">

# **7. End of the Notebook**

[Back to TOP](#TOP)
</a>

From the modeling and assessment stage, we developed and evaluated several predictive models to estimate car prices based on the processed features. Different algorithms were trained, and their performance was compared using appropriate evaluation metrics to identify the most effective approach. The analysis of results provided valuable insights into the model’s predictive capabilities, highlighting strengths, limitations, and potential areas for improvement. This stage represents a crucial step in validating the overall workflow and ensuring that the final model delivers accurate and reliable predictions..